In [ ]:
# --------------------------------------------------------------------
# 1. Imports and settings
# --------------------------------------------------------------------
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta
pd.set_option('display.max_columns', 100)

In [ ]:
# --------------------------------------------------------------------
# 2. Load the essential CSV files
# --------------------------------------------------------------------
# We read transaction.csv and transaction_v2.csv
# The dates in transactions.csv are parsed automatically.
# --------------------------------------------------------------------
DATA_DIR = "csv_files/"

dtypes = {
    'msno': 'string',             
    'payment_method_id': 'Int16',    
    'payment_plan_days': 'Int16',   
    'plan_list_price': 'Int16',     
    'actual_amount_paid': 'Int16', 
    'is_auto_renew': 'Int8',     
    'is_cancel': 'Int8',           
}

transactions_v1 = pd.read_csv(
    os.path.join(DATA_DIR, "transactions.csv"),
    dtype = dtypes,
    parse_dates=[
        "transaction_date",
        "membership_expire_date"
    ]
)

transactions_v2 = pd.read_csv(
    os.path.join(DATA_DIR, "transactions_v2.csv"),
    dtype = dtypes,
    parse_dates=[
        "transaction_date",
        "membership_expire_date"
    ]
)

transactions = pd.concat([transactions_v1, transactions_v2], ignore_index=True)

print("Transactions shape:", transactions.shape)

In [ ]:
#custom functions for generating the cohort data

def calculate_renewal_gap(group):
    last_expire_date = group.last_expire.iloc[0]

    gap = 9999

    for _, row in group.iterrows():
        trans_date = row["transaction_date"]
        expire_date = row["membership_expire_date"]
        if(expire_date == datetime(1970, 1, 1)):
            expire_date= trans_date

        is_cancel = row["is_cancel"]

        if is_cancel == 1:
            if expire_date < last_expire_date:
                last_expire_date = expire_date
        else:
            gap = (trans_date - last_expire_date).days
            break

    return gap

def sort_transactions(df):
    temp_df = df.copy()

    # Convert dates to numeric for mathematical inversion
    temp_df["expire_numeric"] = temp_df["membership_expire_date"].astype("int64")

    # 1. Group check: count how many cancellations are in this day's pair (0, 1, or 2)
    temp_df["cancel_sum"] = temp_df.groupby(["msno", "transaction_date"])[
        "is_cancel"
    ].transform("sum")

    # 2. Group check: are plan_list_price AND payment_plan_days identical for the pair?
    temp_df["same_details"] = (
        temp_df.groupby(["msno", "transaction_date"])["plan_list_price"].transform(
            "nunique"
        )
        == 1
    ) & (
        temp_df.groupby(["msno", "transaction_date"])[
            "payment_plan_days"
        ].transform("nunique")
        == 1
    )

    # TIE BREAKER 1: For the "One True, One False" scenario
    # If details match: False (0) then True (1) -> sort by is_cancel directly
    # If details differ: True (1) then False (0) -> invert it (-is_cancel)
    temp_df["tie_breaker_1"] = np.where(
        temp_df["same_details"], temp_df["is_cancel"], -temp_df["is_cancel"]
    )

    # TIE BREAKER 2: For the "Both True" or "Both False" scenarios
    # Both True (cancel_sum == 2): Earliest expire comes last -> Inverted (-expire_numeric)
    # Both False (cancel_sum == 0): Latest expire comes last -> Normal (expire_numeric)
    temp_df["tie_breaker_2"] = np.where(
        temp_df["cancel_sum"] == 2,
        -temp_df["expire_numeric"],
        temp_df["expire_numeric"],
    )

    # Execute the master sort hierarchy
    df_sorted = temp_df.sort_values(
        by=["msno", "transaction_date", "tie_breaker_1", "tie_breaker_2"],
        ascending=[True, True, True, True],
    )

    # Clean up all temporary helper columns
    df_sorted = df_sorted.drop(
        columns=[
            "expire_numeric",
            "cancel_sum",
            "same_details",
            "tie_breaker_1",
            "tie_breaker_2",
        ]
    )

    return df_sorted

def cohort_transaction_data_generator(sorted_trans_data: pd.DataFrame, month, year):
    history_cutoff = datetime(year, month, 1)

    #We break the data into history and future data based on history_cutoff
    history_data = sorted_trans_data[sorted_trans_data['transaction_date'] < history_cutoff].copy()
    future_data = sorted_trans_data[(sorted_trans_data["transaction_date"] >= history_cutoff) & (sorted_trans_data['transaction_date'] <= history_cutoff + timedelta(60))].copy()

    second_to_last_transaction = history_data.groupby('msno', as_index=False).nth(-2)
    last_transaction = history_data.groupby('msno', as_index = False).last()

    this_month_cancel_users = last_transaction.loc[(last_transaction['membership_expire_date'] >= (history_cutoff - timedelta(30))) &
                                                   (last_transaction['membership_expire_date'] < history_cutoff) &
                                                   (last_transaction['is_cancel'] == 1)].msno.unique()
    greater_than_this_month_users = second_to_last_transaction.loc[second_to_last_transaction['membership_expire_date'] >= history_cutoff].msno.unique()

    greater_than_this_month_canceled_users = np.intersect1d(this_month_cancel_users, greater_than_this_month_users)
    greater_than_this_month_canceled = last_transaction.loc[last_transaction.msno.isin(greater_than_this_month_canceled_users)][['msno', 'membership_expire_date']]

    next_month = history_cutoff
    if(month == 12):
        next_month = datetime(year + 1, 1, 1)
    else:
        next_month = datetime(year, month + 1, 1)

    this_month_expire = last_transaction.loc[
    (last_transaction["membership_expire_date"] >= history_cutoff)
    & (last_transaction["membership_expire_date"] < next_month)
    ][['msno', 'membership_expire_date']]

    prediction_candidates = pd.concat([greater_than_this_month_canceled, this_month_expire], ignore_index = True)
    prediction_candidates.columns = ['msno', 'last_expire']

    # Left outer join with future data
    joined_data = pd.merge(
    prediction_candidates, future_data, on="msno", how="left"
    )

    no_activity = joined_data[joined_data["payment_method_id"].isna()].copy()
    no_activity["is_churn"] = 1

    has_activity = joined_data[joined_data["payment_method_id"].notna()].copy()
    renewal_gap = (
    has_activity.groupby(["msno"])
    .apply(calculate_renewal_gap, include_groups=False)
    .reset_index(name = 'gap')
    )

    valid_renewals = renewal_gap[renewal_gap["gap"] <= 30].copy()
    valid_renewals["is_churn"] = 0

    late_renewals = renewal_gap[renewal_gap["gap"] > 30].copy()
    late_renewals["is_churn"] = 1

    churn_label =  pd.concat(
    [
        valid_renewals[["msno", "is_churn"]],
        late_renewals[["msno", "is_churn"]],
        no_activity[["msno", "is_churn"]],
    ],
    ignore_index=True,
    )
    modified_last_transaction = last_transaction.copy()
    modified_last_transaction.drop(columns = ['payment_method_id'], inplace = True)
    modified_last_transaction.columns = ['msno', 'last_payment_plan_days',
                                         'last_plan_list_price',
                                         'last_actual_amount_paid',
                                         'last_is_auto_renew',
                                         'last_transaction_date',
                                         'last_expire', 'last_is_cancel']

    trans_agg = history_data.groupby('msno').agg(
    num_transactions=('transaction_date', 'count'),
    total_paid=('actual_amount_paid', 'sum'),
    avg_plan_price=('plan_list_price', 'mean'),
    total_auto_renew=('is_auto_renew', 'sum'),
    total_cancel=('is_cancel', 'sum'),
    first_transaction_date=('transaction_date', 'min')
    ).reset_index()

    trans_agg = pd.merge(churn_label, trans_agg, on = 'msno', how = 'left')
    trans_agg = pd.merge(trans_agg, modified_last_transaction, on = 'msno', how = 'left')
    trans_agg['days_since_first_trans'] = (history_cutoff - trans_agg['first_transaction_date']).dt.days
    trans_agg['days_since_last_trans'] = (history_cutoff - trans_agg['last_transaction_date']).dt.days
    trans_agg['days_to_expire'] = (trans_agg['last_expire']- history_cutoff).dt.days
    trans_agg['avg_payment_per_day'] = trans_agg['total_paid'] / ((trans_agg['days_since_first_trans'] + trans_agg['days_to_expire']))

    trans_agg.drop(columns=['first_transaction_date'], inplace=True)

    return trans_agg



In [ ]:
data_day_agg = transactions.groupby(['msno', 'transaction_date'], as_index = False).agg(num_transactions = ('payment_method_id', 'count'))
mult_users = data_day_agg.loc[data_day_agg.num_transactions > 2].msno.unique()
transactions = transactions[~transactions["msno"].isin(mult_users)]
sorted_transactions = sort_transactions(transactions)

In [ ]:
#Now let's generate cohort data

import os
import pandas as pd

# 1. Define a worker function for a single (month, year) combination
def process_cohort(month, year):
    print(f"Processing: {month}/{year}")
    
    # Generate the data
    cohort_data = cohort_transaction_data_generator(sorted_transactions, month, year)
    cohort_data['cohort_cutoff_date'] = datetime(year, month, 1)
    
    return cohort_data

#2. Run a loop to get all the desired cohorts
results = []
for year in range(2015, 2018):
    for month in range(1, 13):
        # Skip condition
        if year == 2015 and month == 1:
            continue
        
        # Break condition
        if year == 2017 and month == 3:
            break
            
        results.append(process_cohort(month, year))

# 3. Combine everything into one final DataFrame
train = pd.concat(results, ignore_index=True)

# 5. Save your result
train.to_csv(
    os.path.join(DATA_DIR, "mult_cohort_transaction_data.csv"), index=False
)
